In [132]:
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
import pickle

In [151]:
data = pd.read_csv('bundesliga_pripravena_data.csv', sep=',')

data['datum_zapasu'] = pd.to_datetime(data['datum_zapasu'], format='%d.%m.%Y', errors='coerce')
data['mesic'] = data['datum_zapasu'].dt.month
data['den_v_tydnu'] = data['datum_zapasu'].dt.dayofweek

In [152]:
input_features = ["forma_tymu_domaci" ,"forma_tymu_hoste", "domaci_dane_goly", "domaci_dostane_goly", "hoste_dane_goly", "hoste_dostane_goly", "domaci_forma_doma", "hoste_forma_venku", "h2h_uspesnost_domacich", "domaci_ocekavane_goly", "hoste_ocekavane_goly", "domaci_forma_brankare", "hoste_forma_brankare", "mesic", "den_v_tydnu","elo_domaci", "elo_hoste", "elo_rozdil"]

Domaci kurz

In [147]:

import numpy as np

data['prob_domaci'] = 1 / data['kurz_domaci']
target_feature_domaci = 'prob_domaci'
X_train, X_test, y_train, y_test = train_test_split(data[input_features], data[target_feature_domaci], test_size=0.3, shuffle=False)

_, _, _, y_test_kurz = train_test_split(
    data[input_features],
    data['kurz_domaci'],
    test_size=0.3,
    shuffle=False
)

model_domaci = xgb.XGBRegressor(n_estimators=1500, learning_rate=0.03, max_depth=7, random_state=32)
model_domaci.fit(X_train, y_train)

predikce_prob = model_domaci.predict(X_test)

predikce_prob = np.clip(predikce_prob, 0.01, 0.99)

predikce_kurz = 1 / predikce_prob

chyba = mean_absolute_error(y_test_kurz, predikce_kurz)

print(f"Nová průměrná chyba odhadu kurzu na domácí: {round(chyba, 2)}")

ukazka = pd.DataFrame({
    'Skutečný kurz bet365': y_test_kurz.values,
    'Náš model': predikce_kurz,
    'Rozdíl': abs(y_test_kurz.values - predikce_kurz)
})
print("\nUkázka:")
print(ukazka.tail().round(2))

chyba_prob = mean_absolute_error(y_test, predikce_prob)

print(f"Model se plete v odhadu šance na výhru v průměru o: {round(chyba_prob * 100, 2)} %")

pickle.dump(model_domaci, open('model_domaci.dat', 'wb'))

Nová průměrná chyba odhadu kurzu na domácí: 0.66

Ukázka:
     Skutečný kurz bet365  Náš model  Rozdíl
820                  1.20       1.25    0.05
821                  1.37       1.32    0.05
822                  3.50       3.33    0.17
823                  1.67       1.30    0.37
824                  1.80       1.69    0.11
Model se plete v odhadu šance na výhru v průměru o: 6.94 %


Remiza kurz

In [150]:
data['prob_remiza'] = 1 / data['kurz_remiza']
target_feature_domaci = 'prob_remiza'
X_train, X_test, y_train, y_test = train_test_split(data[input_features], data[target_feature_domaci], test_size=0.3, shuffle=False)

_, _, _, y_test_kurz = train_test_split(
    data[input_features],
    data['kurz_remiza'],
    test_size=0.3,
    shuffle=False
)

model_remiza = xgb.XGBRegressor(n_estimators=1500, learning_rate=0.01, max_depth=2, random_state=32)
model_remiza.fit(X_train, y_train)

predikce_prob = model_remiza.predict(X_test)

predikce_prob = np.clip(predikce_prob, 0.01, 0.99)

predikce_kurz = 1 / predikce_prob

chyba = mean_absolute_error(y_test_kurz, predikce_kurz)

print(f"Nová průměrná chyba odhadu kurzu na remíza: {round(chyba, 2)}")

ukazka = pd.DataFrame({
    'Skutečný kurz bet365': y_test_kurz.values,
    'Náš model': predikce_kurz,
    'Rozdíl': abs(y_test_kurz.values - predikce_kurz)
})
print("\nUkázka:")
print(ukazka.tail().round(2))

chyba_prob = mean_absolute_error(y_test, predikce_prob)

print(f"Model se plete v odhadu šance na výhru v průměru o: {round(chyba_prob * 100, 2)} %")

pickle.dump(model_remiza, open('model_remiza.dat', 'wb'))

Nová průměrná chyba odhadu kurzu na remíza: 1.46

Ukázka:
     Skutečný kurz bet365  Náš model  Rozdíl
820                  12.0      12.54    0.54
821                   9.5      10.80    1.30
822                   8.5       8.98    0.48
823                   8.0       9.37    1.37
824                   8.0       8.44    0.44
Model se plete v odhadu šance na výhru v průměru o: 1.11 %


Kurz hoste

In [153]:
data['prob_hoste'] = 1 / data['kurz_hoste']
target_feature_domaci = 'prob_hoste'
X_train, X_test, y_train, y_test = train_test_split(data[input_features], data[target_feature_domaci], test_size=0.3, shuffle=False)

_, _, _, y_test_kurz = train_test_split(
    data[input_features],
    data['kurz_hoste'],
    test_size=0.3,
    shuffle=False
)

model_hoste = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=7, random_state=32)
model_hoste.fit(X_train, y_train)

predikce_prob = model_hoste.predict(X_test)

predikce_prob = np.clip(predikce_prob, 0.01, 0.99)

predikce_kurz = 1 / predikce_prob

chyba = mean_absolute_error(y_test_kurz, predikce_kurz)

print(f"Nová průměrná chyba odhadu kurzu na hosté: {round(chyba, 2)}")

ukazka = pd.DataFrame({
    'Skutečný kurz bet365': y_test_kurz.values,
    'Náš model': predikce_kurz,
    'Rozdíl': abs(y_test_kurz.values - predikce_kurz)
})
print("\nUkázka:")
print(ukazka.tail().round(2))

chyba_prob = mean_absolute_error(y_test, predikce_prob)

print(f"Model se plete v odhadu šance na výhru v průměru o: {round(chyba_prob * 100, 2)} %")

pickle.dump(model_hoste, open('model_hoste.dat', 'wb'))

Nová průměrná chyba odhadu kurzu na hosté: 1.28

Ukázka:
     Skutečný kurz bet365  Náš model  Rozdíl
820                  7.50       6.02    1.48
821                  4.50       4.70    0.20
822                  1.52       1.65    0.13
823                  3.00       4.97    1.97
824                  2.70       3.36    0.66
Model se plete v odhadu šance na výhru v průměru o: 6.72 %
